# 4周年企画: 今年よく見られた「1年目の動画」

2026-08-12 のシクフォニ4周年に向けて、**1年目（2022-08-12 〜 2023-08-11）に投稿された動画**が
**今年（2026年）どれだけ見られたか**をランキング化し、再生リストを手がかりにジャンル別へ集約する。

| 出力 | 内容 |
|---|---|
| ランキング表 | 投稿日 / タイトル / URL / 今年の増加再生数 / 累計 |
| ジャンル別集計 | 再生リスト所属から分類したジャンルごとの本数・増加数シェア |
| CSV | 上記をまとめた `sixfonia_anniversary_year1_<日付>.csv` |

**指標の定義**: 期間内の最初と最後のスナップショットの差（`metrics.period_gains`）。
日次差分の合計ではないので、収集が飛んだ日があっても増加量を取りこぼさない。

**注意**: 再生リスト所属はAPIから新規取得する（既存CSVには無い）。公開プレイリストのみ取得可。
SQL版（BigQuery移行後用）は `anniversary_year1_ranking.sql` を参照。

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

In [ ]:
#@title ⚙️ 期間・対象の設定
CHANNEL = "sixfonia"  #@param ["sixfonia", "hima72", "kosame", "illuma", "mikoto", "suchi", "lan"]

#@markdown 1年目の範囲（チャンネル開設日 〜 1周年前日）
YEAR1_START = "2022-08-12"  #@param {type:"string"}
YEAR1_END   = "2023-08-11"  #@param {type:"string"}

#@markdown 「今年」として集計する範囲
WINDOW_START = "2026-01-01"  #@param {type:"string"}
WINDOW_END   = "2026-08-11"  #@param {type:"string"}

TOP_N = 20  #@param {type:"integer"}

import pandas as pd
from sixfonia_analytics import auth, collect, config, display as sfx_display, load, metrics, playlists

pd.set_option("display.max_colwidth", 60)
print(f"対象: {config.CHANNEL_BY_NAME[CHANNEL]['display']}  "
      f"1年目 {YEAR1_START}〜{YEAR1_END} / 集計 {WINDOW_START}〜{WINDOW_END}")

In [ ]:
#@title 📥 日次統計のロードとデータ期間の確認
combined = load.build_combined_df(CHANNEL)

dmin, dmax = combined["view_date"].min(), combined["view_date"].max()
print(f"データ期間: {dmin.date()} 〜 {dmax.date()}")

# 集計期間がデータで埋まっているか確認（欠測が多いと増加量が過小になる）
win = combined[(combined["view_date"] >= WINDOW_START) & (combined["view_date"] <= WINDOW_END)]
n_days = win["view_date"].nunique()
expected = (pd.Timestamp(min(WINDOW_END, str(dmax.date()))) - pd.Timestamp(WINDOW_START)).days + 1
print(f"集計期間内の収集日数: {n_days} 日 / 想定 {expected} 日")
if n_days < expected * 0.8:
    print("⚠️ 欠測が多い。端点差方式なので合計値への影響は小さいが、期間の端が"
          "実際より内側に寄っていないか first_date / last_date を確認すること")

In [ ]:
#@title 📈 今年の増加再生数を算出（pandas版クエリ）
gains = metrics.period_gains(combined, WINDOW_START, WINDOW_END)
print(f"対象動画: {len(gains)} 本")
gains.head(5)

In [ ]:
#@title 🎬 動画マスタを結合して1年目の動画に絞り込み
#@markdown 投稿日・タイトル・動画長は動画マスタ（`01_collect/video_master` の出力）から取る。
#@markdown 無ければAPIから取得する。
master_path = config.channel_dir(CHANNEL) / f"{CHANNEL}_video_master.csv"
if master_path.exists():
    master = pd.read_csv(master_path)
    print(f"保存済みマスタを使用: {master_path}")
else:
    print("保存済みマスタがないためAPIから取得します...")
    youtube = auth.build_youtube()
    master = pd.DataFrame(collect.fetch_video_master(youtube, config.channel_id_of(CHANNEL)))
    collect.save_master_csv(master.to_dict("records"), CHANNEL)

master["published_at"] = pd.to_datetime(master["published_at"], utc=True).dt.tz_convert("Asia/Tokyo")
master["published_date"] = master["published_at"].dt.tz_localize(None).dt.normalize()

year1 = master[
    (master["published_date"] >= YEAR1_START) & (master["published_date"] <= YEAR1_END)
].copy()
print(f"1年目の投稿本数: {len(year1)} 本")

rank_df = (
    year1.rename(columns={"video_id": "videoId", "title": "video_title"})
    .merge(gains, on="videoId", how="inner")
)
rank_df["videoURL"] = "https://www.youtube.com/watch?v=" + rank_df["videoId"]
rank_df["video_type"] = rank_df["duration_seconds"].apply(metrics.classify_video_type)
rank_df["publishedAt"] = rank_df["published_date"]
rank_df["thumbnailURL"] = rank_df["thumbnail_url"]
rank_df = rank_df.sort_values("views_gained", ascending=False).reset_index(drop=True)
rank_df["rank_no"] = rank_df.index + 1

missing = len(year1) - len(rank_df)
if missing:
    print(f"⚠️ {missing} 本が統計側に見つからない（非公開化・削除の可能性）")
print(f"ランキング対象: {len(rank_df)} 本")

In [ ]:
#@title 🏆 総合ランキング（今年の増加再生数）
sfx_display.display_ranking_table(
    rank_df,
    title=f"今年よく見られた1年目の動画 TOP {TOP_N}（{WINDOW_START}〜{WINDOW_END}の増加再生数）",
    metric_col="views_gained",
    top_n=TOP_N,
)

cols = ["rank_no", "published_date", "video_title", "videoURL",
        "views_gained", "views_at_end", "gain_share_pct", "video_type"]
display(rank_df[cols].head(TOP_N))

In [ ]:
#@title 🔥 「今も伸びている」動画（累計に占める今年分の割合）
#@markdown 母数が小さい動画はノイズになるので下限を設ける
MIN_TOTAL_VIEWS = 10000  #@param {type:"integer"}

rediscovered = (
    rank_df[rank_df["views_at_end"] >= MIN_TOTAL_VIEWS]
    .sort_values("gain_share_pct", ascending=False)
    .head(TOP_N)
)
display(rediscovered[["published_date", "video_title", "videoURL",
                      "views_gained", "views_at_end", "gain_share_pct"]])

In [ ]:
#@title 📚 再生リスト所属の取得（初回のみAPIを叩く）
#@markdown 取得結果は `<channel>_playlist_map.csv` に保存され、次回以降は再利用される。
#@markdown プレイリストを編集した後は REFRESH にチェックを入れて再取得すること。
REFRESH = False  #@param {type:"boolean"}

youtube = auth.build_youtube()
pl_map = playlists.build_playlist_map(youtube, CHANNEL, refresh=REFRESH)

print("\n【プレイリスト別の収録本数】")
display(pl_map["playlist_title"].value_counts().to_frame("videos"))

In [ ]:
#@title 🏷️ ジャンル分類（再生リスト名ベース）
#@markdown 分類ルールは `sixfonia_analytics/playlists.py` の `GENRE_RULES`。
#@markdown 実際のプレイリスト名を見て調整する。ここで一時的に上書きもできる。

# 例: ルールをこのnotebook内で差し替える場合
# custom_rules = [("歌ってみた", [r"歌って", r"cover"]), ("企画", [r"企画", r"検証"])]
custom_rules = None

genred = playlists.add_genre(rank_df, pl_map, id_col="videoId",
                             title_col="video_title", rules=custom_rules)

print("【ジャンル別本数】")
display(genred["genre"].value_counts().to_frame("videos"))

print("\n【未分類の確認 → ここを見て GENRE_RULES を足す】")
display(playlists.unclassified_report(genred))

In [ ]:
#@title 📊 ジャンル別サマリ
summary = playlists.genre_summary(genred, metric_col="views_gained")
display(summary)

import matplotlib.pyplot as plt
from sixfonia_analytics import plots

plots.setup_japanese_font()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(summary["genre"][::-1], summary["total"][::-1], color="#1f77b4")
axes[0].set_title(f"ジャンル別 今年の増加再生数（{WINDOW_START}〜{WINDOW_END}）")
axes[0].set_xlabel("増加再生数")
axes[0].grid(axis="x", linestyle="--", alpha=0.6)

axes[1].barh(summary["genre"][::-1], summary["video_count"][::-1], color="#4ecdc4")
axes[1].set_title("ジャンル別 1年目の投稿本数")
axes[1].set_xlabel("本数")
axes[1].grid(axis="x", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎖️ ジャンル別ランキング（各ジャンルのTOP N）
PER_GENRE_TOP = 5  #@param {type:"integer"}

for genre in summary["genre"]:
    sub = genred[genred["genre"] == genre].sort_values("views_gained", ascending=False)
    sfx_display.display_ranking_table(
        sub, title=f"【{genre}】 {len(sub)}本中 TOP {min(PER_GENRE_TOP, len(sub))}",
        metric_col="views_gained", top_n=PER_GENRE_TOP,
    )

In [ ]:
#@title 💾 CSV出力
export_cols = [
    "rank_no", "genre", "published_date", "video_title", "videoURL",
    "playlists_str", "video_type", "duration_seconds",
    "views_gained", "views_at_start", "views_at_end", "gain_share_pct",
    "first_date", "last_date", "videoId",
]
out = genred.sort_values("views_gained", ascending=False)[export_cols]
out = out.rename(columns={
    "rank_no": "順位", "genre": "ジャンル", "published_date": "投稿日",
    "video_title": "動画タイトル", "videoURL": "動画URL",
    "playlists_str": "所属再生リスト", "video_type": "種別",
    "views_gained": "今年の増加再生数", "views_at_end": "累計再生数",
    "gain_share_pct": "累計に占める今年分[%]",
})

out_path = config.channel_dir(CHANNEL) / f"{CHANNEL}_anniversary_year1_{WINDOW_END.replace('-', '')}.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"保存: {out_path}  ({len(out)} 行)")
display(out.head(10))